# BTC: FuzzyLSTM vs SOTA (PatchTST/TimesNet/TFT/TiDE) + Naive + Walk-forward

Этот ноутбук делает:
- Шаг 1: добавляет Naive baselines (по цене и по log-return).
- Шаг 2: делает сопоставимый multi-step для FuzzyLSTM через recursive forecasting.
- Шаг 4: оценивает модели на 3 walk-forward окнах (2–3 окна).

Окружение: Google Colab (GPU для PyTorch/NeuralForecast, TensorFlow принудительно на CPU).


In [1]:
# --- Colab: уменьшение фрагментации CUDA памяти для PyTorch ---
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ВАЖНО: после выполнения этой ячейки сделай Runtime -> Restart runtime, затем запускай всё сверху вниз.


In [2]:
# Установка зависимостей (в Colab можно запускать один раз)
!pip -q install -U pip
!pip -q install -U yfinance openpyxl
!pip -q install -U neuralforecast pytorch-lightning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.6 MB/s eta 0:00:00


In [3]:
import os, gc, time, math
import numpy as np
import pandas as pd

# --- TensorFlow на CPU (чтобы не занимал GPU память) ---
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
from tensorflow.keras import backend as K
from keras.layers import Dense, RNN, Input
from tensorflow.keras.layers import LSTMCell
from keras.initializers import Constant

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import yfinance as yf
from datetime import date, timedelta

import torch
import matplotlib.pyplot as plt

# --- NeuralForecast ---
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import PatchTST, TimesNet, TFT, TiDE

def rmse(a,b):
    return float(np.sqrt(mean_squared_error(a,b)))

print("torch cuda available:", torch.cuda.is_available())


torch cuda available: False


In [4]:
def get_historical_close_data(name,step):
  if step=='1d':
    start = "1900-01-01"
    end = date.today()
    data = yf.Ticker(name)
    data=data.history(start=start, end=end,interval ='1d')
    data.reset_index(inplace=True)
    data = data.loc[:,('Date','Close')]
    return data
  if step=='1h':
    start = date.today()-timedelta(days=729)
    end = date.today()
    data = yf.Ticker(name)
    data=data.history(start=start, end=end,interval ='1h')
    data.reset_index(inplace=True)
    data = data.loc[:,('Datetime','Close')]
    return data
  if step=='1m':
    start = date.today()-timedelta(days=5)
    end = date.today()
    data = yf.Ticker(name)
    data = data.history(start=start, end=end, interval='1m')
    data.reset_index(inplace=True)
    # yfinance обычно отдаёт 'Datetime' для внутридневных интервалов
    if 'Datetime' in data.columns:
        data = data.loc[:, ('Datetime','Close')]
    elif 'Date' in data.columns:
        data = data.loc[:, ('Date','Close')]
    return data

In [5]:
def split_train_test(data):

    # split into train and test sets
    train_size = int(len(data) * 0.75)
    train, test = data[0:train_size,], data[train_size:len(data),]
    return train, test

def create_dataset(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset)-lookback-1):
      a = dataset[i:(i+lookback), 0]
      dataX.append(a)
      dataY.append(dataset[i + lookback, 0])
    return np.array(dataX), np.array(dataY)


In [6]:
def create_dataset_multifeature_1(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset)-lookback-1):
        a = dataset[i:(i+lookback), :]     # теперь берем все фичи
        dataX.append(a)
        dataY.append(dataset[i + lookback, 0])  # целевая переменная — всё так же первая (норм. цена)
    return np.array(dataX), np.array(dataY)

def create_dataset_multifeature(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset) - lookback - 1):
        a = dataset[i:(i + lookback), :]
        dataX.append(a)
        dataY.append(dataset[i + lookback, :])  # теперь y — вектор из 2 значений
    return np.array(dataX), np.array(dataY)


In [7]:

class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        # Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  # предыдущий скрытый
        c_tm1 = states[1]  # предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        # Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     # input gate
        f = self.param_sigmoid(z1)     # forget gate
        c = f * c_tm1 + i * K.tanh(z2) # cell state
        o = self.param_sigmoid(z3)     # output gate

        h = o * K.tanh(c)
        return h, [h, c]



In [8]:
def fit_fuzzy_onestep_model(close_df: pd.DataFrame, lookback:int=120, units:int=32):
    """Обучаем one-step модель на (price_norm, phi) -> next price_norm.
    phi строим каузально по изменениям и rolling std последних 50 изменений."""
    scaler = MinMaxScaler(feature_range=(0,1))
    y = close_df["Close"].values.reshape(-1,1)
    y_norm = scaler.fit_transform(y).astype("float32").flatten()

    # каузальный phi
    dy = np.diff(y_norm, prepend=y_norm[0])
    win_std = 50
    sigma = pd.Series(dy).rolling(win_std).std().shift(1).bfill().fillna(1e-3).values
    sigma = np.maximum(sigma, 1e-4)
    phi = np.exp(-(dy**2)/(2*(sigma**2))).astype("float32")

    X = np.column_stack([y_norm, phi]).astype("float32")
    # dataset -> (samples, lookback, 2), target -> y_norm next
    Xs, ys = create_dataset_multifeature_1(X, lookback)
    Xs = Xs.reshape((Xs.shape[0], lookback, 2)).astype("float32")
    ys = ys.astype("float32")

    inp = Input(shape=(lookback,2))
    h = RNN(ParametricLSTMCell_sigmoid(units), name="fuzzy_cell")(inp)
    out = Dense(1, name="yhat")(h)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

    model.fit(Xs, ys, epochs=8, batch_size=128, verbose=0)
    return model, scaler

def fuzzy_recursive_forecast(model, scaler, train_close: np.ndarray, h:int, lookback:int=120):
    """Рекурсивный multi-step прогноз: на каждом шаге предсказываем next price_norm и дописываем в ряд."""
    # train_close: prices (float) длины >= lookback+2
    y_norm = scaler.transform(train_close.reshape(-1,1)).astype("float32").flatten().tolist()

    def compute_phi_series(y_norm_list):
        dy = np.diff(np.array(y_norm_list), prepend=y_norm_list[0])
        win_std = 50
        sigma = pd.Series(dy).rolling(win_std).std().shift(1).bfill().fillna(1e-3).values
        sigma = np.maximum(sigma, 1e-4)
        phi = np.exp(-(dy**2)/(2*(sigma**2))).astype("float32")
        return phi.tolist()

    for _ in range(h):
        phi = compute_phi_series(y_norm)
        # последние lookback точек
        x_win = np.column_stack([y_norm[-lookback:], phi[-lookback:]]).astype("float32")[None,:,:]
        yhat_norm = float(model.predict(x_win, verbose=0).reshape(-1)[0])
        y_norm.append(yhat_norm)

    yhat_norm_arr = np.array(y_norm[-h:], dtype="float32").reshape(-1,1)
    yhat = scaler.inverse_transform(yhat_norm_arr).flatten()
    return yhat


In [9]:
def naive_price_forecast(last_price: float, h:int):
    return np.full(h, last_price, dtype="float64")

def naive_logreturn_forecast(last_price: float, h:int):
    # r_hat=0 => цена плоская
    return np.full(h, last_price, dtype="float64")


In [10]:
def make_nf_df(close_df: pd.DataFrame, uid="BTC-USD"):
    """Формат NeuralForecast: unique_id, ds, y.
    Поддерживает yfinance колонки Date/Datetime и случай, когда время в индексе."""
    df = close_df.copy()

    # Если время в индексе — вынесем
    if 'ds' not in df.columns and df.index.name is not None:
        if any(k in str(df.index.name).lower() for k in ['date','time','datetime']):
            df = df.reset_index()

    # Время
    if 'ds' not in df.columns:
        for cand in ['Datetime','Date','date','datetime','timestamp','Time']:
            if cand in df.columns:
                df = df.rename(columns={cand:'ds'})
                break

    # Цель
    if 'y' not in df.columns:
        for cand in ['Close','close','Adj Close','adj_close','y']:
            if cand in df.columns:
                df = df.rename(columns={cand:'y'})
                break

    missing = [c for c in ['ds','y'] if c not in df.columns]
    if missing:
        raise RuntimeError(f"Не найдено {missing}. Колонки: {list(df.columns)}")

    df['unique_id'] = uid
    df['ds'] = pd.to_datetime(df['ds'])
    return df[['unique_id','ds','y']].sort_values('ds').reset_index(drop=True)
def nf_predict_one_window(train_df: pd.DataFrame, freq: str, h:int, input_size:int, max_steps:int,
                          batch_size:int=8, windows_batch_size:int=32, inference_windows_batch_size:int=32,
                          model_name:str="PatchTST"):
    common = dict(h=h, input_size=input_size, loss=MAE(), max_steps=max_steps,
                  batch_size=batch_size, windows_batch_size=windows_batch_size,
                  inference_windows_batch_size=inference_windows_batch_size)
    if model_name=="PatchTST":
        model = PatchTST(**common)
    elif model_name=="TimesNet":
        model = TimesNet(**common)
    elif model_name=="TFT":
        model = TFT(**common)
    elif model_name=="TiDE":
        model = TiDE(**common)
    else:
        raise ValueError(model_name)

    nf = NeuralForecast(models=[model], freq=freq)
    nf.fit(df=train_df)
    fcst = nf.predict(df=train_df)  # прогноз на следующие h после конца train
    # cleanup
    del nf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return fcst


In [11]:
def evaluate_window(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
    }

def run_walk_forward(step:str, n_windows:int=3, max_steps:int=200, lookback:int=120, units:int=32):
    ticker="BTC-USD"
    close_df = get_historical_close_data(ticker, step)
    close_df = close_df.dropna().reset_index(drop=True)

    # horizon / input_size presets
    if step=="1m":
        h=60; input_size=240; freq="1min"
    elif step=="1h":
        h=24; input_size=240; freq="1H"
    elif step=="1d":
        h=14; input_size=140; freq="D"
    else:
        raise ValueError(step)

    y_all = close_df["Close"].values.astype("float64")
    results = []
    for w in range(n_windows):
        train_end = len(y_all) - (n_windows - w)*h
        train_end = max(train_end, lookback+5)
        train_prices = y_all[:train_end]
        test_prices = y_all[train_end:train_end+h]
        if len(test_prices) < h:
            continue

        # --- Naive baselines ---
        last_price = float(train_prices[-1])
        for m in ["NaivePrice","NaiveLogReturn"]:
            yhat = naive_price_forecast(last_price, h) if m=="NaivePrice" else naive_logreturn_forecast(last_price, h)
            met = evaluate_window(test_prices, yhat)
            results.append({"step":step,"window":w,"model":m, **met})

        # --- FuzzyLSTM recursive ---
        # train model on train slice (CPU)
        train_df = close_df.iloc[:train_end].copy()
        model_fz, scaler_fz = fit_fuzzy_onestep_model(train_df, lookback=lookback, units=units)
        yhat_fz = fuzzy_recursive_forecast(model_fz, scaler_fz, train_prices, h=h, lookback=lookback)
        met = evaluate_window(test_prices, yhat_fz)
        results.append({"step":step,"window":w,"model":"FuzzyLSTM_recursive", **met})
        del model_fz
        tf.keras.backend.clear_session()
        gc.collect()

        # --- NeuralForecast models ---
        nf_train = make_nf_df(train_df, uid=ticker)
        for m in ["PatchTST","TimesNet","TFT","TiDE"]:
            fcst = nf_predict_one_window(
                train_df=nf_train, freq=freq, h=h, input_size=input_size, max_steps=max_steps,
                batch_size=8, windows_batch_size=32, inference_windows_batch_size=32,
                model_name=m
            )
            # fcst contains columns: unique_id, ds, <model>
            yhat = fcst[m].values.astype("float64")
            met = evaluate_window(test_prices, yhat)
            results.append({"step":step,"window":w,"model":m, **met})
    return pd.DataFrame(results)



In [12]:
# --- Запуск: BTC-USD, 3 окна walk-forward ---
ALL = []
for step in ["1m","1h","1d"]:
    print("="*90)
    print(f"STEP={step} | BTC-USD")
    print("="*90)
    df_res = run_walk_forward(step=step, n_windows=3, max_steps=200, lookback=120, units=32)
    display(df_res.head())
    ALL.append(df_res)

res = pd.concat(ALL, ignore_index=True)

# агрегирование по окнам
agg = (res
       .groupby(["step","model"], as_index=False)
       .agg(RMSE_mean=("RMSE","mean"), MAE_mean=("MAE","mean"), MAPE_mean=("MAPE","mean"),
            RMSE_std=("RMSE","std"))
      ).sort_values(["step","RMSE_mean"])

print("\n=== Итог (среднее по окнам) ===")
display(agg)

# сохранить в Excel
out_path = "BTC_walkforward_results.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    res.to_excel(w, index=False, sheet_name="per_window")
    agg.to_excel(w, index=False, sheet_name="summary")

print("Saved:", out_path)


STEP=1m | BTC-USD


INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE               │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d     │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm      │      0 │ train │     0 │
│ 3 │ model        │ PatchTST_backbone │  633 K │ train │     0 │
└───┴──────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 633 K                                                                                            
Non-trainable params: 3                                                                                            
Total params: 633 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 90                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()